# 03 — KPI Calculations & Feature Engineering
**Logistics Operations Analytics Dashboard**

---

This notebook covers:
1. Feature engineering (derived columns)
2. KPI calculations across all dimensions
3. Customer retention cohort analysis
4. Driver performance scoring
5. Export KPI summaries for Power BI & Excel

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

PALETTE = {
    'primary': '#2563eb', 'success': '#10b981',
    'warning': '#f59e0b', 'danger': '#ef4444',
    'purple': '#8b5cf6'
}

print('Ready for KPI calculations!')

In [ ]:
# Load data
cleaned = '../data/logistics_orders_cleaned.csv'
raw = '../data/logistics_orders.csv'
df = pd.read_csv(cleaned if os.path.exists(cleaned) else raw)
df['order_date'] = pd.to_datetime(df['order_date'])
print(f'Loaded: {df.shape}')

---
## 1. Feature Engineering

In [ ]:
# ─── Temporal Features ───────────────────────────────────────────────────────
df['order_hour'] = pd.to_datetime(df['order_time'], format='%H:%M:%S').dt.hour
df['month'] = df['order_date'].dt.month
df['month_name'] = df['order_date'].dt.strftime('%Y-%m')
df['quarter'] = df['order_date'].dt.quarter
df['year'] = df['order_date'].dt.year
df['day_of_week'] = df['order_date'].dt.day_name()
df['is_weekend'] = df['order_date'].dt.dayofweek >= 5

# Hour bucket
def hour_bucket(h):
    if 6 <= h < 12: return 'Morning'
    elif 12 <= h < 17: return 'Afternoon'
    elif 17 <= h < 22: return 'Evening'
    else: return 'Night'

df['order_hour_bucket'] = df['order_hour'].apply(hour_bucket)

# ─── Delivery Performance Features ───────────────────────────────────────────
df['delivery_delay_min'] = df['actual_delivery_min'] - df['estimated_delivery_min']
df['is_delayed'] = df['delivery_delay_min'] > 0
df['delay_severity'] = pd.cut(
    df['delivery_delay_min'],
    bins=[-np.inf, -5, 0, 5, 15, np.inf],
    labels=['Early (>5min)', 'On-Time', 'Slight Delay', 'Moderate Delay', 'Severe Delay']
)

# ─── Revenue Features ────────────────────────────────────────────────────────
df['revenue_per_km'] = np.where(
    df['distance_km'] > 0,
    (df['total_revenue'] / df['distance_km']).round(2),
    0
)

# ─── Customer Features ───────────────────────────────────────────────────────
df['customer_order_rank'] = df.groupby('customer_id').cumcount() + 1

print('Engineered features:')
new_cols = ['order_hour', 'month_name', 'quarter', 'year', 'day_of_week', 'is_weekend',
            'order_hour_bucket', 'delivery_delay_min', 'is_delayed', 'delay_severity',
            'revenue_per_km', 'customer_order_rank']
for col in new_cols:
    print(f'  + {col}')
print(f'\nNew shape: {df.shape}')

---
## 2. Overall KPI Dashboard

In [ ]:
# ─── Calculate all KPIs ──────────────────────────────────────────────────────
delivered = df[df['order_status'] == 'Delivered']

kpis = {
    'Total Orders': len(df),
    'Delivered Orders': len(delivered),
    'Total Revenue (INR)': df['total_revenue'].sum(),
    'Total Revenue (Cr)': df['total_revenue'].sum() / 1e7,
    'Average Order Value (INR)': df['order_value'].mean(),
    'Average Delivery Fee (INR)': df['delivery_fee'].mean(),
    'Average Delivery Time (min)': delivered['actual_delivery_min'].mean(),
    'Median Delivery Time (min)': delivered['actual_delivery_min'].median(),
    'On-Time Delivery %': (delivered['delivery_delay_min'] <= 0).mean() * 100,
    'Cancellation Rate %': (df['order_status'] == 'Cancelled').mean() * 100,
    'Return Rate %': (df['order_status'] == 'Returned').mean() * 100,
    'Average Rating': df['rating'].mean(),
    'Average Distance (km)': df['distance_km'].mean(),
    'Revenue per km (INR)': delivered['revenue_per_km'].mean(),
    'Unique Customers': df['customer_id'].nunique(),
    'Unique Drivers': df['driver_id'].nunique(),
    'Cities Covered': df['city'].nunique(),
    'Repeat Customer %': df['is_repeat_customer'].mean() * 100,
    'Orders per Driver (avg)': len(df) / df['driver_id'].nunique(),
    'Revenue per Driver (INR)': df['total_revenue'].sum() / df['driver_id'].nunique(),
}

kpi_df = pd.DataFrame(list(kpis.items()), columns=['KPI', 'Value'])
kpi_df['Value'] = kpi_df['Value'].round(2)

print('='*55)
print('LOGISTICS OPERATIONS — KPI SUMMARY')
print('='*55)
for _, row in kpi_df.iterrows():
    print(f"  {row['KPI']:<35} {row['Value']:>15,.2f}")

---
## 3. City-Wise KPIs

In [ ]:
city_kpis = df.groupby('city').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('total_revenue', 'sum'),
    avg_order_value=('order_value', 'mean'),
    avg_delivery_time=('actual_delivery_min', 'mean'),
    avg_rating=('rating', 'mean'),
    cancel_rate=('order_status', lambda x: (x == 'Cancelled').mean() * 100),
    on_time_pct=('is_delayed', lambda x: (~x).mean() * 100),
    unique_customers=('customer_id', 'nunique'),
    active_drivers=('driver_id', 'nunique'),
    avg_distance=('distance_km', 'mean'),
    repeat_customer_pct=('is_repeat_customer', 'mean')
).round(2)

city_kpis['revenue_share_pct'] = (city_kpis['total_revenue'] / city_kpis['total_revenue'].sum() * 100).round(2)
city_kpis['repeat_customer_pct'] = (city_kpis['repeat_customer_pct'] * 100).round(2)
city_kpis = city_kpis.sort_values('total_revenue', ascending=False)

print('City-Wise KPIs (Top 15):')
city_kpis.head(15)

---
## 4. Monthly KPIs & Trends

In [ ]:
monthly_kpis = df.groupby('month_name').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('total_revenue', 'sum'),
    avg_order_value=('order_value', 'mean'),
    avg_delivery_time=('actual_delivery_min', 'mean'),
    cancel_rate=('order_status', lambda x: (x == 'Cancelled').mean() * 100),
    avg_rating=('rating', 'mean'),
    unique_customers=('customer_id', 'nunique'),
    new_customers=('customer_order_rank', lambda x: (x == 1).sum()),
    repeat_orders=('customer_order_rank', lambda x: (x > 1).sum())
).round(2)

# MoM Growth
monthly_kpis['revenue_mom_growth'] = monthly_kpis['total_revenue'].pct_change() * 100
monthly_kpis['order_mom_growth'] = monthly_kpis['total_orders'].pct_change() * 100

print('Monthly KPIs:')
monthly_kpis.round(2)

---
## 5. Driver Performance Scoring

In [ ]:
driver_kpis = df.groupby(['driver_id', 'driver_name']).agg(
    total_trips=('order_id', 'count'),
    deliveries=('order_status', lambda x: (x == 'Delivered').sum()),
    cancellations=('order_status', lambda x: (x == 'Cancelled').sum()),
    total_revenue=('total_revenue', 'sum'),
    avg_delivery_time=('actual_delivery_min', 'mean'),
    avg_rating=('rating', 'mean'),
    total_distance=('distance_km', 'sum'),
    on_time_pct=('is_delayed', lambda x: (~x).mean() * 100)
).round(2)

# Driver utilization score (composite)
driver_kpis['delivery_rate'] = (driver_kpis['deliveries'] / driver_kpis['total_trips'] * 100).round(2)
driver_kpis['revenue_per_trip'] = (driver_kpis['total_revenue'] / driver_kpis['total_trips']).round(2)

# Composite performance score (0-100)
from sklearn.preprocessing import MinMaxScaler
try:
    scaler = MinMaxScaler()
    score_cols = ['avg_rating', 'on_time_pct', 'delivery_rate', 'total_trips']
    scores = scaler.fit_transform(driver_kpis[score_cols].fillna(0))
    weights = [0.35, 0.30, 0.20, 0.15]
    driver_kpis['performance_score'] = (np.average(scores, weights=weights, axis=1) * 100).round(1)
except ImportError:
    # Fallback without sklearn
    for col in ['avg_rating', 'on_time_pct', 'delivery_rate', 'total_trips']:
        col_min = driver_kpis[col].min()
        col_max = driver_kpis[col].max()
        driver_kpis[f'{col}_norm'] = (driver_kpis[col] - col_min) / (col_max - col_min)
    driver_kpis['performance_score'] = (
        driver_kpis['avg_rating_norm'] * 35 +
        driver_kpis['on_time_pct_norm'] * 30 +
        driver_kpis['delivery_rate_norm'] * 20 +
        driver_kpis['total_trips_norm'] * 15
    ).round(1)
    driver_kpis = driver_kpis.drop(columns=[c for c in driver_kpis.columns if c.endswith('_norm')])

driver_kpis = driver_kpis.sort_values('performance_score', ascending=False)
driver_kpis['rank'] = range(1, len(driver_kpis) + 1)

print('Top 20 Drivers by Performance Score:')
driver_kpis.head(20)

---
## 6. Customer Retention Analysis

In [ ]:
# Customer segmentation by order frequency
cust_summary = df.groupby('customer_id').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('total_revenue', 'sum'),
    avg_order_value=('order_value', 'mean'),
    first_order=('order_date', 'min'),
    last_order=('order_date', 'max'),
    avg_rating_given=('rating', 'mean'),
    cities_ordered=('city', 'nunique')
).round(2)

# Customer segments
def segment(orders):
    if orders == 1: return 'One-Time'
    elif orders <= 3: return 'Returning'
    elif orders <= 6: return 'Regular'
    elif orders <= 10: return 'Loyal'
    else: return 'VIP'

cust_summary['segment'] = cust_summary['total_orders'].apply(segment)
cust_summary['customer_lifetime_days'] = (cust_summary['last_order'] - cust_summary['first_order']).dt.days

# Segment summary
seg_summary = cust_summary.groupby('segment').agg(
    customer_count=('total_orders', 'count'),
    avg_orders=('total_orders', 'mean'),
    avg_ltv=('total_revenue', 'mean'),
    total_revenue=('total_revenue', 'sum'),
    avg_lifetime_days=('customer_lifetime_days', 'mean')
).round(2)

seg_summary['customer_pct'] = (seg_summary['customer_count'] / seg_summary['customer_count'].sum() * 100).round(2)
seg_summary['revenue_pct'] = (seg_summary['total_revenue'] / seg_summary['total_revenue'].sum() * 100).round(2)

# Reorder
seg_order = ['One-Time', 'Returning', 'Regular', 'Loyal', 'VIP']
seg_summary = seg_summary.reindex(seg_order)

print('Customer Segmentation:')
seg_summary

In [ ]:
# Visualize customer segments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = [PALETTE['danger'], PALETTE['warning'], PALETTE['primary'],
          PALETTE['success'], PALETTE['purple']]

# Customer count by segment
bars = axes[0].bar(seg_summary.index, seg_summary['customer_count'],
                   color=colors, alpha=0.85, edgecolor='white')
axes[0].set_title('Customers by Segment', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].bar_label(bars, fmt='{:,.0f}', fontsize=10)

# Revenue by segment
bars = axes[1].bar(seg_summary.index, seg_summary['total_revenue'] / 1e5,
                   color=colors, alpha=0.85, edgecolor='white')
axes[1].set_title('Revenue by Segment (Lakhs)', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Revenue (Lakhs INR)')
axes[1].bar_label(bars, fmt='{:,.0f}L', fontsize=10)

plt.tight_layout()
plt.savefig('../data/kpi_segments.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Vehicle Type Performance

In [ ]:
vehicle_kpis = df.groupby('vehicle_type').agg(
    total_orders=('order_id', 'count'),
    total_revenue=('total_revenue', 'sum'),
    avg_order_value=('order_value', 'mean'),
    avg_delivery_time=('actual_delivery_min', 'mean'),
    avg_distance=('distance_km', 'mean'),
    avg_rating=('rating', 'mean'),
    cancel_rate=('order_status', lambda x: (x == 'Cancelled').mean() * 100),
    on_time_pct=('is_delayed', lambda x: (~x).mean() * 100),
    avg_delivery_fee=('delivery_fee', 'mean')
).round(2)

vehicle_kpis['revenue_per_km'] = (vehicle_kpis['total_revenue'] / 
                                    df.groupby('vehicle_type')['distance_km'].sum()).round(2)
vehicle_kpis['order_share_pct'] = (vehicle_kpis['total_orders'] / vehicle_kpis['total_orders'].sum() * 100).round(2)

print('Vehicle Type KPIs:')
vehicle_kpis

---
## 8. Export KPI Data

In [ ]:
# ─── Export for Power BI ──────────────────────────────────────────────────────
powerbi_path = '../powerbi/powerbi_data.csv'
os.makedirs('../powerbi', exist_ok=True)
df.to_csv(powerbi_path, index=False)
print(f'Power BI data exported: {powerbi_path}')

# ─── Export KPI summaries ─────────────────────────────────────────────────────
os.makedirs('../excel', exist_ok=True)

# Export all KPI tables
kpi_df.to_csv('../data/kpi_overall.csv', index=False)
city_kpis.to_csv('../data/kpi_city.csv')
monthly_kpis.to_csv('../data/kpi_monthly.csv')
driver_kpis.to_csv('../data/kpi_drivers.csv')
seg_summary.to_csv('../data/kpi_customer_segments.csv')
vehicle_kpis.to_csv('../data/kpi_vehicle.csv')

print('\nAll KPI exports complete!')
print('Files exported:')
for f in ['kpi_overall', 'kpi_city', 'kpi_monthly', 'kpi_drivers',
          'kpi_customer_segments', 'kpi_vehicle']:
    print(f'  - data/{f}.csv')
print(f'  - powerbi/powerbi_data.csv')

In [ ]:
# ─── Generate Excel Workbook ──────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

wb = Workbook()

# Style definitions
header_font = Font(name='Calibri', bold=True, size=11, color='FFFFFF')
header_fill = PatternFill(start_color='2563EB', end_color='2563EB', fill_type='solid')
alt_fill = PatternFill(start_color='F0F4FF', end_color='F0F4FF', fill_type='solid')
title_font = Font(name='Calibri', bold=True, size=14, color='0F172A')
border = Border(
    left=Side(style='thin', color='D1D5DB'),
    right=Side(style='thin', color='D1D5DB'),
    top=Side(style='thin', color='D1D5DB'),
    bottom=Side(style='thin', color='D1D5DB')
)

def write_df_to_sheet(ws, df_input, title, start_row=1):
    """Write a DataFrame to an Excel sheet with professional formatting."""
    # Title
    ws.cell(row=start_row, column=1, value=title).font = title_font
    start_row += 2
    
    # Reset index if needed
    if df_input.index.name or not isinstance(df_input.index, pd.RangeIndex):
        df_input = df_input.reset_index()
    
    # Headers
    for col_idx, col_name in enumerate(df_input.columns, 1):
        cell = ws.cell(row=start_row, column=col_idx, value=str(col_name))
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal='center')
        cell.border = border
    
    # Data rows
    for row_idx, row in enumerate(df_input.itertuples(index=False), start_row + 1):
        for col_idx, value in enumerate(row, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.border = border
            cell.alignment = Alignment(horizontal='center')
            if (row_idx - start_row) % 2 == 0:
                cell.fill = alt_fill
    
    # Auto-width columns
    for col_idx, col_name in enumerate(df_input.columns, 1):
        max_len = max(len(str(col_name)), df_input[col_name].astype(str).str.len().max())
        ws.column_dimensions[chr(64 + col_idx) if col_idx <= 26 else 'A'].width = min(max_len + 4, 30)
    
    # Freeze panes
    ws.freeze_panes = ws.cell(row=start_row + 1, column=1)


# Sheet 1: KPI Summary
ws1 = wb.active
ws1.title = 'KPI Summary'
write_df_to_sheet(ws1, kpi_df, 'Logistics Operations — KPI Summary')

# Sheet 2: City Performance
ws2 = wb.create_sheet('City Performance')
write_df_to_sheet(ws2, city_kpis.head(25), 'City-Wise Performance Metrics')

# Sheet 3: Monthly Trends
ws3 = wb.create_sheet('Monthly Trends')
write_df_to_sheet(ws3, monthly_kpis, 'Monthly Revenue & Order Trends')

# Sheet 4: Driver Scorecard
ws4 = wb.create_sheet('Driver Scorecard')
write_df_to_sheet(ws4, driver_kpis.head(50), 'Driver Performance Scorecard')

# Sheet 5: Customer Segments
ws5 = wb.create_sheet('Customer Segments')
write_df_to_sheet(ws5, seg_summary, 'Customer Retention Segments')

# Sheet 6: Vehicle Analysis
ws6 = wb.create_sheet('Vehicle Analysis')
write_df_to_sheet(ws6, vehicle_kpis, 'Vehicle Type Performance')

# Save
excel_path = '../excel/logistics_summary.xlsx'
wb.save(excel_path)
print(f'\nExcel workbook saved: {excel_path}')
print(f'Sheets: {wb.sheetnames}')

In [ ]:
print('='*60)
print('KPI CALCULATIONS COMPLETE')
print('='*60)
print(f'\nFeatures engineered: 12 new columns')
print(f'KPI tables generated: 6')
print(f'Files exported: 8 (6 CSV + 1 Excel + 1 Power BI CSV)')
print(f'\nReady for Power BI dashboard build!')